# 02 — Batch preprocess (v2)

Pipeline `raw → processed` pour TOUS les participants × TOUTES les conditions.

**Nouveautés v2** :
- Lit `trial.crop_start_s` et `trial.crop_end_s` depuis `participants.py`
- Trim des trajectoires AVANT preprocess (donc avant EMD)
- Skip automatique des essais `excluded=True`
- Sélection auto des IMFs (bande [0.5–2.5 Hz]) — voir `preprocess.py`

**Comportement** :
- `FORCE=True` → retraite tout (écrase les `.npz` existants)
- `FORCE=False` → skip les `.npz` déjà présents
- Erreurs par essai = log warning, continue la boucle (résumé en fin)

Pour mise au point sur 1 participant, voir `01_preprocess_one.ipynb`.

In [2]:
from resilience import paths, participants, config
from resilience.io import loader, writer
from resilience.processing import preprocess
import time

## Configuration

In [3]:
FORCE      = True   # True = retraite tout / False = skip les .npz existants

# Toute la liste des 39 essais à traiter (auto via excluded=False)
TRIALS = participants.list_trials_to_process()

print(f"Total essais à traiter : {len(TRIALS)}")
print(f"FORCE = {FORCE}\n")

# Affichage compact des essais
from collections import defaultdict
by_p = defaultdict(list)
for p, c in TRIALS:
    by_p[p].append(c)
for p in sorted(by_p):
    print(f"  {p:10s} → {len(by_p[p])} essai(s)  ({', '.join(by_p[p])})")

Total essais à traiter : 39
FORCE = True

  001CrMa    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  002CrPa    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  003BrLu    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  004CaGe    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  005DeJe    → 2 essai(s)  (beatmove_adaptatif, tempo_random)
  006MoCa    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  007MoMa    → 1 essai(s)  (tempo_random)
  008RiMo    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  009DeFr    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  010DeYv    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  011RiJo    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  012WaCh    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  013WaJe    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)
  014DeAn    → 3 essai(s)  (beatmove_adaptatif, silence, tempo_random)


## Boucle batch

In [4]:
results = {'ok': [], 'skipped': [], 'failed': []}
t_start = time.time()
current_p = None

for participant, condition in TRIALS:
    if participant != current_p:
        print(f"\n{'═'*60}\n  {participant}\n{'═'*60}")
        current_p = participant

    key = f"{participant} / {condition}"
    try:
        # 1. Vérifier si .npz existe déjà
        if not FORCE:
            try:
                npz = paths.processed_file(participant, condition)
                print(f"  ⏭️  {condition:22s} → skip (existe : {npz.name})")
                results['skipped'].append(key)
                continue
            except FileNotFoundError:
                pass

        # 2. Trouver le .mat brut
        raw_path = paths.raw_file(participant, condition)
        print(f"  ▶️   {condition:22s} → {raw_path.name}")

        # 3. Load + extraction trajectoires
        data, fmt = loader.load_mat(raw_path)
        trajectories = loader.extract_marker_trajectories(data, fmt)
        if trajectories is None:
            raise RuntimeError("extract_marker_trajectories a retourné None")

        # 3bis. CROPS manuels (crop_start_s + crop_end_s) depuis participants.py
        trial = participants.get_trial(participant, condition)
        n_before = next(iter(trajectories.values())).shape[0]
        i_start = 0
        i_end   = n_before

        if trial.crop_start_s is not None:
            i_start = int(trial.crop_start_s * config.FS_RAW)
            print(f"      ✂️  crop_start_s = {trial.crop_start_s}s")
        if trial.crop_end_s is not None:
            i_end = int(trial.crop_end_s * config.FS_RAW)
            print(f"      ✂️  crop_end_s   = {trial.crop_end_s}s")

        if i_start > 0 or i_end < n_before:
            trajectories = {m: xyz[i_start:i_end] for m, xyz in trajectories.items()}
            n_after = i_end - i_start
            print(f"      → {n_before} frames → {n_after} frames ({n_after/config.FS_RAW:.1f}s)")

        # 4. Preprocess (EMD auto + Butterworth + décimation)
        out = preprocess.run(trajectories, axis='Z', verbose=False)

        # 5. Sauvegarde .npz
        npz_path = writer.save_processed(participant, condition, out,
                                          trajectories_raw=trajectories)
        print(f"      ✅ {npz_path.name}")
        results['ok'].append(key)

    except FileNotFoundError as e:
        print(f"      ⚠️  fichier .mat introuvable")
        results['failed'].append((key, 'FileNotFoundError', str(e)))
    except Exception as e:
        print(f"      ❌ {type(e).__name__}: {e}")
        results['failed'].append((key, type(e).__name__, str(e)))

elapsed = time.time() - t_start
print(f"\n{'═'*60}")
print(f"  Temps total : {elapsed/60:.1f} min  ({elapsed:.0f} s)")
print(f"{'═'*60}")


════════════════════════════════════════════════════════════
  001CrMa
════════════════════════════════════════════════════════════
  ▶️   beatmove_adaptatif     → 001CrMa_03_beatmove_adaptatif.mat
      ✂️  crop_start_s = 60.0s
      ✂️  crop_end_s   = 358s
      → 192001 frames → 119200 frames (298.0s)
      ✅ 001CrMa_03_beatmove_adaptatif.npz
  ▶️   silence                → 001CrMa_01_silence.mat
      ✂️  crop_start_s = 60.0s
      → 192000 frames → 168000 frames (420.0s)
      ✅ 001CrMa_01_silence.npz
  ▶️   tempo_random           → 001CrMa_02_tempo_random.mat
      ✂️  crop_start_s = 60.0s
      → 192000 frames → 168000 frames (420.0s)
      ✅ 001CrMa_02_tempo_random.npz

════════════════════════════════════════════════════════════
  002CrPa
════════════════════════════════════════════════════════════
  ▶️   beatmove_adaptatif     → 002CrPa_02_beatmove_adaptatif.mat
      ✂️  crop_start_s = 120s
      → 192000 frames → 144000 frames (360.0s)
      ✅ 002CrPa_02_beatmove_adaptatif

## Résumé

In [5]:
print(f"\n  ✅ OK       : {len(results['ok'])}")
print(f"  ⏭️  Skipped  : {len(results['skipped'])}")
print(f"  ❌ Failed   : {len(results['failed'])}")

if results['failed']:
    print(f"\n  Détail des échecs :")
    for key, err_type, err_msg in results['failed']:
        print(f"    [{err_type}] {key}")
        print(f"        → {err_msg[:120]}")

# Essais explicitement EXCLUS (non traités) :
excluded = [(p.code, cond) for p in participants.PARTICIPANTS.values()
            for cond, t in p.trials.items() if t.excluded]
if excluded:
    print(f"\n  🚫 Essais exclus (excluded=True) : {len(excluded)}")
    for p, c in excluded:
        trial = participants.get_trial(p, c)
        print(f"    {p} / {c}  —  {trial.note or 'sans note'}")


  ✅ OK       : 39
  ⏭️  Skipped  : 0
  ❌ Failed   : 0

  🚫 Essais exclus (excluded=True) : 3
    005DeJe / silence  —  EXCLUDE: artefacts 180-230s, baseline pré-CRF non récupérable
    007MoMa / silence  —  EXCLUDE: capteurs dos cassés, bug Codamotion Odin
    007MoMa / beatmove_adaptatif  —  EXCLUDE: signal globalement dégradé sur tout l'essai


## Vérification rapide d'un .npz produit

In [6]:
import numpy as np

if results['ok']:
    p, c = results['ok'][0].split(' / ')
    payload = writer.load_processed(p, c)
    print(f"  Fichier : {p} / {c}")
    print(f"\n  Clés présentes :")
    for k in payload:
        arr = payload[k]
        if isinstance(arr, np.ndarray):
            print(f"    {k:22s} shape={str(arr.shape):20s} dtype={arr.dtype}")
        else:
            print(f"    {k:22s} = {arr}")

  Fichier : 001CrMa / beatmove_adaptatif

  Clés présentes :
    sacrum_xyz_raw         shape=(119200, 3)          dtype=float64
    sacrum_axis_raw        shape=(119200,)            dtype=float64
    sacrum_emd             shape=(119200,)            dtype=float64
    sacrum_filt            shape=(119200,)            dtype=float64
    signal_final           shape=(29800,)             dtype=float64
    time_final             shape=(29800,)             dtype=float64
    fs_final               shape=()                   dtype=int64
    axis                   shape=()                   dtype=<U1
    emd_applied            shape=()                   dtype=bool
    participant            shape=()                   dtype=<U7
    condition              shape=()                   dtype=<U18
    raw_Dos01              shape=(119200, 3)          dtype=float64
    raw_Dos02              shape=(119200, 3)          dtype=float64
    raw_Dos03              shape=(119200, 3)          dtype=float64
   